# MeguvizCAC v0.4-DualLoop — mHC + LOTUS + CharRhythm + GroupDRO + 1.4M Corpus

Implements all empirical post-mortem + external OOD generalization upgrades over **v0.3-Loop**:
- **CPU Fusion & Selective Step-LoRA**: native ORT `LayerNormalization` + external step scale/shift + selective `Q/V/O + FFN_down` LoRA (targeting `<= 2.8 ms/row` unloaded)
- **Dual-Teacher & LOTUS 4-Slot Internalization**: Qwen3.8-27B pragmatic/intent + Gemini 4 literary/code + Born-Again EMA self-distillation
- **mHC (N=4) + CharRhythmStem + Optional DiffAttn**: pre-baked doubly-stochastic stream mixing + bounded step-residual damping + 1D UTF-8 byte Conv side-stem
- **Anti-Regression Deep Loops & OOD Precision**: monotonic step-loss penalty + late-step LoRA L2 + Asymmetric Tail-Push (`R@P95%`/`P99%`) + 11-GroupDRO (incl. `structured_human_hard`) + zero-PII `@`/URL domain coverage + PAVA/Conformal + Ordered W4-Emb Edge


## 1. GPU Check, Repo Setup & Ordered-W4 Dependencies


In [ ]:
!nvidia-smi
import os, torch
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3) if torch.cuda.is_available() else 0.0
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"Detected GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB | bf16: {torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False}")

REPO_DIR = "/content/MeguvizCAC"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/jichuo1/MeguvizCAC.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd /content/MeguvizCAC
!git log -1 --oneline
!pip -q install -U onnx onnx_ir ml_dtypes onnxruntime transformers scipy scikit-learn


## 2. Download Stage-1/2/3 Assets & v0.4 1.25M+ Corpus Bundle


In [ ]:
import os
!mkdir -p /content/data/stage2 /content/data/v03_teachers /content/data/v04

!curl -fL "https://github.com/jichuo1/MeguvizCAC/releases/download/v0.1-data/bili-hydrogen-rescored-350387.jsonl.gz" -o /content/data/bili-hydrogen-rescored-350387.jsonl.gz
!curl -fL "https://github.com/jichuo1/MeguvizCAC/releases/download/v0.2-data/stage2_distill_bundle.zip" -o /content/data/stage2_distill_bundle.zip
!unzip -qo /content/data/stage2_distill_bundle.zip -d /content/data/stage2

!curl -fL "https://github.com/jichuo1/MeguvizCAC/releases/download/v0.3.0/meguvizcac-v0.3-loop-2x4-e192-int8.zip" -o /content/data/v03_teachers/e192.zip
!curl -fL "https://github.com/jichuo1/MeguvizCAC/releases/download/v0.3.0/meguvizcac-v0.3-loop-2x4-e128-int8.zip" -o /content/data/v03_teachers/e128.zip
!unzip -qo /content/data/v03_teachers/e192.zip -d /content/data/v03_teachers/e192
!unzip -qo /content/data/v03_teachers/e128.zip -d /content/data/v03_teachers/e128

V04_REAL_URL = "https://github.com/jichuo1/MeguvizCAC/releases/download/v0.4-data/bili-v04-real-1m.jsonl.gz"
V04_SYNTH_URL = "https://github.com/jichuo1/MeguvizCAC/releases/download/v0.4-data/bili-v04-synthetic-250k.jsonl.gz"
if not os.path.isfile("/content/data/v04/bili-v04-real-1m.jsonl.gz"):
    !curl -fL "{V04_REAL_URL}" -o /content/data/v04/bili-v04-real-1m.jsonl.gz || echo "NOTE: Upload/mount bili-v04-real-1m.jsonl.gz into /content/data/v04/"
if not os.path.isfile("/content/data/v04/bili-v04-synthetic-250k.jsonl.gz"):
    !curl -fL "{V04_SYNTH_URL}" -o /content/data/v04/bili-v04-synthetic-250k.jsonl.gz || echo "NOTE: Upload/mount bili-v04-synthetic-250k.jsonl.gz into /content/data/v04/"
!ls -lh /content/data /content/data/v04


## 3. GPU 12L Teacher Pseudo-Labeling + Intent Cache + Unified 1.4M Corpus Merge & Dry-Run


In [ ]:
!python -m meguviz_cac.v04_teacher_score \
  --real-jsonl /content/data/v04/bili-v04-real-1m.jsonl.gz \
  --stage1-data /content/data/bili-hydrogen-rescored-350387.jsonl.gz \
  --qwen-fused /content/data/stage2/stage2-fused-train.jsonl.gz \
  --qwen-distill /content/data/stage2/qwen38_27b_distill.jsonl \
  --synth-path /content/data/v04/bili-v04-synthetic-250k.jsonl.gz \
  --train-12l-teacher --teacher-backbone hfl/chinese-macbert-base --teacher-epochs 2 \
  --build-intent-cache /content/data/v04_semantic_intent_cache.npz \
  --out-scores /content/data/v04-teacher-scores.jsonl.gz \
  --active-pool-out /content/data/v04-active-audit-pool.jsonl

!python -m meguviz_cac.v04_corpus_prep \
  --stage1-data /content/data/bili-hydrogen-rescored-350387.jsonl.gz \
  --qwen-data /content/data/stage2/stage2-fused-train.jsonl.gz \
  --qwen-distill /content/data/stage2/qwen38_27b_distill.jsonl \
  --pairs-data /content/data/stage2/stage2-literary-pairs.jsonl \
  --v04-synth /content/data/v04/bili-v04-synthetic-250k.jsonl.gz \
  --v04-real-jsonl /content/data/v04/bili-v04-real-1m.jsonl.gz \
  --teacher-scores /content/data/v04-teacher-scores.jsonl.gz \
  --export-out /content/data/v04-merged-1m4.jsonl.gz

!python -m meguviz_cac.train_v04_dualloop --dry-run \
  --data /content/data/v04-merged-1m4.jsonl.gz \
  --qwen-data /content/data/stage2/stage2-fused-train.jsonl.gz \
  --pairs-data /content/data/stage2/stage2-literary-pairs.jsonl \
  --intent-cache /content/data/v04_semantic_intent_cache.npz \
  --vocab /content/data/v03_teachers/e192/vocab.txt


## 4. 60-Second End-to-End Smoke Test (3-Stage + Baked mHC + Dual ONNX + Ordered W4-Emb Edge)


In [ ]:
!python -m meguviz_cac.train_v04_dualloop --smoke-test \
  --data /content/data/v04-merged-1m4.jsonl.gz \
  --qwen-data /content/data/stage2/stage2-fused-train.jsonl.gz \
  --pairs-data /content/data/stage2/stage2-literary-pairs.jsonl \
  --intent-cache /content/data/v04_semantic_intent_cache.npz \
  --version v0.4-smoke \
  --out /content/out_v04_smoke


## 5. Primary Model: `v0.4-dualloop-2x4-e192` (`mHC-4`, `LOTUS-4`, `CharStem`, `Fused LN`, `Selective LoRA`)


In [ ]:
!python -m meguviz_cac.train_v04_dualloop \
  --data /content/data/v04-merged-1m4.jsonl.gz \
  --qwen-data /content/data/stage2/stage2-fused-train.jsonl.gz \
  --pairs-data /content/data/stage2/stage2-literary-pairs.jsonl \
  --intent-cache /content/data/v04_semantic_intent_cache.npz \
  --version v0.4-dualloop-2x4-e192 \
  --embed-dim 192 --core-layers 2 --num-loops 4 --normal-max-loops 3 \
  --num-latents 4 --mhc-streams 4 --lora-rank 16 --lora-targets qvo_down \
  --epochs 2 --refine-epochs 1 --qat-epochs 2 --batch-size 0 \
  --out /content/out_v04_2x4_e192


## 6. OOD Generalization Ablation: `v0.4-dualloop-2x4-e128` (`E=128`, `mHC-4`, `LOTUS-4`, `CharStem`)


In [ ]:
!python -m meguviz_cac.train_v04_dualloop \
  --data /content/data/v04-merged-1m4.jsonl.gz \
  --qwen-data /content/data/stage2/stage2-fused-train.jsonl.gz \
  --pairs-data /content/data/stage2/stage2-literary-pairs.jsonl \
  --intent-cache /content/data/v04_semantic_intent_cache.npz \
  --version v0.4-dualloop-2x4-e128 \
  --embed-dim 128 --core-layers 2 --num-loops 4 --normal-max-loops 3 \
  --num-latents 4 --mhc-streams 4 --lora-rank 16 --lora-targets qvo_down \
  --epochs 2 --refine-epochs 1 --qat-epochs 2 --batch-size 0 \
  --out /content/out_v04_2x4_e128


## 7. Differential Attention Ablation: `v0.4-dualloop-2x4-e192-diff` (+ Paired DiffAttn)


In [ ]:
!python -m meguviz_cac.train_v04_dualloop \
  --data /content/data/v04-merged-1m4.jsonl.gz \
  --qwen-data /content/data/stage2/stage2-fused-train.jsonl.gz \
  --pairs-data /content/data/stage2/stage2-literary-pairs.jsonl \
  --intent-cache /content/data/v04_semantic_intent_cache.npz \
  --version v0.4-dualloop-2x4-e192-diff \
  --embed-dim 192 --core-layers 2 --num-loops 4 --normal-max-loops 3 \
  --num-latents 4 --mhc-streams 4 --lora-rank 16 --lora-targets qvo_down --diff-attn \
  --epochs 2 --refine-epochs 1 --qat-epochs 2 --batch-size 0 \
  --out /content/out_v04_2x4_e192_diff


## 8. Ultralight Edge Model: `v0.4-dualloop-1x5-e128` (`1x5`, `E=128`, `mHC-1`, `CharStem`)


In [ ]:
!python -m meguviz_cac.train_v04_dualloop \
  --data /content/data/v04-merged-1m4.jsonl.gz \
  --qwen-data /content/data/stage2/stage2-fused-train.jsonl.gz \
  --pairs-data /content/data/stage2/stage2-literary-pairs.jsonl \
  --intent-cache /content/data/v04_semantic_intent_cache.npz \
  --version v0.4-dualloop-1x5-e128 \
  --embed-dim 128 --core-layers 1 --num-loops 5 --normal-max-loops 4 \
  --num-latents 4 --mhc-streams 1 --lora-rank 16 --lora-targets qvo_down \
  --epochs 2 --refine-epochs 1 --qat-epochs 2 --batch-size 0 \
  --out /content/out_v04_1x5_e128


## 9. Cross-Version Comparison (`v0.1` -> `v0.4`), Release Gate Scorecard & Download Packaging


In [ ]:
import json, zipfile
from pathlib import Path

baselines = [
    ("v0.1 (4L bge-small INT8)", 0.9552, None, None, None, 0.8537, 10.67, 0.8667, None, 23.94, 4.0, 9.73, "N/A", "N/A"),
    ("v0.2 (4L Dual-Teacher INT8)", 0.9530, None, None, None, 0.8598, 10.57, 0.9167, None, 23.94, 4.0, 9.86, "N/A", "N/A"),
    ("v0.3-loop-2x4-e192", 0.9565, 0.9406, None, None, 0.8487, 11.10, 0.9000, 0.7879, 14.71, 1.49, 2.09, "skip", "N/A"),
]
run_dirs = [
    Path("/content/out_v04_2x4_e192"),
    Path("/content/out_v04_2x4_e128"),
    Path("/content/out_v04_2x4_e192_diff"),
    Path("/content/out_v04_1x5_e128"),
]

hdr = (
    f"{'Model':30s} | {'AUROC':6s} | {'PRAUC':6s} | {'R@P95':6s} | {'R@P99':6s} | {'Spear':6s} | "
    f"{'MAE':5s} | {'Probe':6s} | {'StHum':6s} | {'MB':5s} | {'ms32':6s} | {'W4':4s} | {'Mono':4s} | Gates"
)
print(hdr)
print("-" * len(hdr))
def _fmt(v, fmt=".4f", dash="-"):
    return format(v, fmt) if v is not None else dash

for name, ta, pra, rp95, rp99, ts, mae, pa, sh, mb, st, ms, edge, mono in baselines:
    print(
        f"{name:30s} | {_fmt(ta)} | {_fmt(pra)} | {_fmt(rp95)} | {_fmt(rp99)} | {_fmt(ts)} | "
        f"{_fmt(mae, '.2f')} | {_fmt(pa)} | {_fmt(sh)} | {mb:5.2f} | {ms:6.2f} | {edge:4s} | {mono:4s} | base"
    )

for d in run_dirs:
    mf = list(d.glob("metrics-*.json"))
    if not mf:
        continue
    m = json.loads(mf[0].read_text(encoding="utf-8"))
    tb = m["test_bili_gold"]
    prb = m["probe_u001_u017"]
    ox = m["onnx_int8"]
    traps = (m.get("onnx_verify") or {}).get("hard_trap_gates") or {}
    edge_ok = "OK" if (m.get("edge_w4_w8") or {}).get("ok") else "FAIL"
    mono_ok = "OK" if (m.get("loop_monotonicity") or {}).get("non_regressive_normal_last_ge_step0") else "WARN"
    rg = m.get("release_gates") or {}
    gate_str = f"{rg.get('passed_count', 0)}/{rg.get('total_gates', 0)}"
    print(
        f"{m['version']:30s} | "
        f"{_fmt(tb.get('auroc'))} | "
        f"{_fmt(tb.get('pr_auc'))} | "
        f"{_fmt(tb.get('recall_at_p95'))} | "
        f"{_fmt(tb.get('recall_at_p99'))} | "
        f"{_fmt(tb.get('spearman'))} | "
        f"{_fmt(tb.get('mae'), '.2f')} | "
        f"{_fmt(prb.get('auroc'))} | "
        f"{_fmt(traps.get('structured_human_max_prob'))} | "
        f"{ox['total_onnx_bytes'] / (1024**2):5.2f} | "
        f"{ox['compact_batch32_1thread_ms']:6.2f} | "
        f"{edge_ok:4s} | {mono_ok:4s} | {gate_str}"
    )

primary_zip = Path("/content/out_v04_2x4_e192/meguvizcac-v0.4-dualloop-2x4-e192-int8.zip")
if primary_zip.is_file():
    print("\n=== v04_runtime Stress, Conformal & Gate Verification ===")
    !python -m meguviz_cac.v04_runtime --bundle {primary_zip}

bundle_out = Path("/content/v04_ablation_bundles.zip")
with zipfile.ZipFile(bundle_out, "w", compression=zipfile.ZIP_STORED) as zf:
    for d in run_dirs:
        for pat in ("meguvizcac-*.zip", "metrics-*.json", "manifest.json"):
            for fp in d.glob(pat):
                zf.write(fp, f"{d.name}/{fp.name}")
    pool_f = Path("/content/data/v04-active-audit-pool.jsonl")
    if pool_f.is_file():
        zf.write(pool_f, pool_f.name)
print(f"Packaged downloadable bundles -> {bundle_out} ({bundle_out.stat().st_size / (1024**2):.2f} MB)")
